In [1]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml

Cloning into 'arabic-itsm-classification'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 130 (delta 47), reused 122 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.59 MiB | 16.93 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/kaggle/working/arabic-itsm-classification
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.6 MB/

In [2]:
# Clear existing placeholder created by git
!rm -rf data/processed && mkdir -p data/processed

# Link using the EXACT path confirmed via ls -R
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/

!ls data/processed
# Expected output: label_encoders.pkl, test.csv, train.csv, val.csv

label_encoders.pkl  test.csv  train.csv  val.csv


In [3]:
!nvidia-smi

Thu Feb 26 02:34:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Joint L1 + L2 + L3 training on Kaggle T4 GPU
# Output: models/marbert_l1_l2_l3_best/  (3 heads: l1, l2, l3)
# This replaces the ambiguous marbert_l3_best (L3-only) as the proper joint L3 checkpoint.

!python scripts/train.py \
    --tasks l1 l2 l3 \
    --data-dir data/processed \
    --epochs 10 \
    --lr 1e-5 \
    --batch-size 16

Device: cuda | FP16: True
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Tasks: ['l1', 'l2', 'l3'] | Classes: {'l1': 6, 'l2': 16, 'l3': 48}
tokenizer_config.json: 100%|███████████████████| 439/439 [00:00<00:00, 2.15MB/s]
vocab.txt: 1.10MB [00:00, 26.5MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 513kB/s]
pytorch_model.bin: 100%|██████████████████████| 654M/654M [00:02<00:00, 265MB/s]
Loading weights:   6%| | 11/199 [00:00<00:00, 1997.03it/s, Materializing param=e
Loading weights: 100%|█| 199/199 [00:00<00:00, 3148.44it/s, Materializing param=
BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                          

In [5]:
import os, shutil

# Verify checkpoint was saved correctly
checkpoint_dir = 'models/marbert_l1_l2_l3_best'
print('Checkpoint files:', os.listdir(checkpoint_dir))

# Verify heads.pt contains the expected 3 tasks
import torch
heads = torch.load(f'{checkpoint_dir}/heads.pt', map_location='cpu', weights_only=False)
print('Head keys:', sorted(heads.keys()))
assert {'l1.1.weight', 'l1.1.bias', 'l2.1.weight', 'l2.1.bias', 'l3.1.weight', 'l3.1.bias'} == set(heads.keys()), \
    f'Unexpected keys: {set(heads.keys())}'
print('\nVerification passed: 3 joint heads found (l1, l2, l3)')

# Create a compressed archive for easy download
shutil.make_archive('/kaggle/working/marbert_l1_l2_l3_best', 'zip', 'models', 'marbert_l1_l2_l3_best')
print('\nDownload archive: /kaggle/working/marbert_l1_l2_l3_best.zip')
print('Place in: arabic-itsm-classification/models/marbert_l1_l2_l3_best/')

Checkpoint files: ['config.json', 'heads.pt', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
Head keys: ['l1.1.bias', 'l1.1.weight', 'l2.1.bias', 'l2.1.weight', 'l3.1.bias', 'l3.1.weight']

Verification passed: 3 joint heads found (l1, l2, l3)

Download archive: /kaggle/working/marbert_l1_l2_l3_best.zip
Place in: arabic-itsm-classification/models/marbert_l1_l2_l3_best/
